In [1]:
# Environment auto-detection (Local Workspace vs. Kaggle)
import os
import numpy as np
import pandas as pd

# Search paths for dataset (checks local repo first)
DATA_SEARCH_PATHS = [
    "student_resource/dataset",
    "dataset",
    "/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset",
    "/kaggle/input/amazon-ml-challenge-2026",
    "/kaggle/input",
]

found_dir = None
for p in DATA_SEARCH_PATHS:
    if os.path.exists(p):
        found_dir = p
        break

if found_dir:
    print(f"Scanning dataset in: {os.path.abspath(found_dir)}")
    for dirname, _, filenames in os.walk(found_dir):
        for filename in sorted(filenames):
            if filename.endswith(".tsv") or filename.endswith(".md"):
                fpath = os.path.join(dirname, filename)
                print(f"  {filename:<26} ({os.path.getsize(fpath)/(1024*1024):.1f} MB)")
else:
    print("No dataset directory found in standard search paths.")

try:
    import kagglehub
except ImportError:
    pass


/kaggle/input/datasets/meerpi/amazon-ml/student_resource/Documentation_template.md
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/README.md
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/utils/validate_submission.py
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_source3.tsv
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_source2.tsv
/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_source1.tsv


In [2]:
!pip install anyascii "sparse-dot-topn>=1.0" -q

import os, gc, re, time
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anyascii
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from tqdm.auto import tqdm  # NEW: live progress bars

print("Libraries successfully imported!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 16.4 MB/s eta 0:00:00
Libraries successfully imported!


In [3]:
def find_data_dirs():
    roots = ["student_resource/dataset", "dataset", "/kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset", "/kaggle/input/amazon-ml-challenge-2026", "/kaggle/input", "."]
    t_dir, te_dir = None, None
    for r in roots:
        if not os.path.exists(r):
            continue
        for dp, _, files in os.walk(r):
            fset = set(files)
            if "test_source1.tsv" in fset and "test_source2.tsv" in fset and te_dir is None:
                te_dir = dp
            if "train_source1.tsv" in fset and "train_ground_truth.tsv" in fset and t_dir is None:
                t_dir = dp
            if t_dir and te_dir:
                break
        if t_dir and te_dir:
            break
    return t_dir, te_dir

TRAIN_DIR, TEST_DIR = find_data_dirs()

# Environment-aware output directories (Local vs Kaggle)
if os.path.exists("/kaggle/working"):
    BASE_OUT = "/kaggle/working"
else:
    BASE_OUT = os.path.abspath("output")

OUTPUT_DIR = os.path.join(BASE_OUT, "submission")
PARTIAL_DIR = os.path.join(BASE_OUT, "partial_test")     # per-country checkpoints for the test run
VAL_PARTIAL_DIR = os.path.join(BASE_OUT, "partial_val")  # per-country checkpoints for validation
for d in (OUTPUT_DIR, PARTIAL_DIR, VAL_PARTIAL_DIR):
    os.makedirs(d, exist_ok=True)

print(f"TRAIN_DIR : {TRAIN_DIR}")
print(f"TEST_DIR  : {TEST_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

test_paths = {
    "s1": os.path.join(TEST_DIR, "test_source1.tsv"),
    "s2": os.path.join(TEST_DIR, "test_source2.tsv"),
    "s3": os.path.join(TEST_DIR, "test_source3.tsv"),
}
train_paths = {
    "s1": os.path.join(TRAIN_DIR, "train_source1.tsv"),
    "s2": os.path.join(TRAIN_DIR, "train_source2.tsv"),
    "s3": os.path.join(TRAIN_DIR, "train_source3.tsv"),
    "gt": os.path.join(TRAIN_DIR, "train_ground_truth.tsv"),
}
for label, paths in [("test", test_paths), ("train", train_paths)]:
    for k, p in paths.items():
        assert os.path.exists(p), f"Missing required {label} file: {p}"
        print(f"  [OK] {label}.{k}: {os.path.getsize(p)/(1024*1024):.1f} MB -> {p}")


TRAIN_DIR : /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train
TEST_DIR  : /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test
  [OK] test.s1: 166.9 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test/test_source1.tsv
  [OK] test.s2: 485.9 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test/test_source2.tsv
  [OK] test.s3: 482.6 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/test/test_source3.tsv
  [OK] train.s1: 200.3 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_source1.tsv
  [OK] train.s2: 466.6 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_source2.tsv
  [OK] train.s3: 480.4 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_source3.tsv
  [OK] train.gt: 121.1 MB -> /kaggle/input/datasets/meerpi/amazon-ml/student_resource/dataset/train/train_ground_truth.tsv


In [4]:
LEGAL_RE = re.compile(
    r'\b(pvt|private|ltd|limited|inc|incorporated|corp|corporation|llc|llp|co|company|sarl|sas|sa|sci|eurl|snc)\b',
    re.IGNORECASE
)
PUNCT_RE = re.compile(r'[^a-z0-9\s]')
SPACE_RE = re.compile(r'\s+')

def normalize_text_list(texts: list) -> list:
    cleaned = []
    for raw in texts:
        if not raw or pd.isna(raw):
            cleaned.append("")
            continue
        t = anyascii.anyascii(str(raw)).lower()
        t = LEGAL_RE.sub(" ", t)
        t = PUNCT_RE.sub(" ", t)
        t = SPACE_RE.sub(" ", t).strip()
        cleaned.append(t)
    return cleaned

def build_composite_text(df: pd.DataFrame) -> list:
    names = df["business_name"].fillna("").astype(str).tolist()
    addrs = df["business_address"].fillna("").astype(str).tolist()
    combined = [f"{n} {a}".strip() for n, a in zip(names, addrs)]
    return normalize_text_list(combined)

# NOTE: if the recall ceiling in Cell 7 looks short, the highest-leverage swap is `uroman`
# for the non-ASCII share of rows (India especially) -- it's real transliteration, not just
# character stripping, and it only costs anything on rows that actually need it. Worth an
# A/B test later, just not a default here since you switched away from LaBSE for speed.

In [5]:
EMPTY_CAND_DF = pd.DataFrame({"entity_id": [], "business_name": [], "business_address": [], "country": []})

def block_one_country(
    country: str,
    sub_s1: pd.DataFrame,
    sub_cand: pd.DataFrame,
    out_path: str,
    k: int = 25,
    threshold: float = 0.10,
    chunk_size: int = 50_000,
    cand_transform_chunk: int = 200_000,  # NEW: chunks the candidate transform purely for progress visibility
    sample_fit_size: int = 250_000,
    n_threads: int = 4,
    seed: int = 42,
):
    # Blocks one country partition and writes source1_entity_id / candidate_entity_ids /
    # scores straight to `out_path`. Skips entirely if out_path already exists and is non-empty.
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"  [{country}] {out_path} already exists -- skipping (delete it to force a rerun)")
        return

    n_queries, n_cands = len(sub_s1), len(sub_cand)
    print(f"  [{country}] Queries: {n_queries:,} | Candidate Pool: {n_cands:,}")

    if n_queries == 0:
        open(out_path, "w").close()
        return

    if n_cands == 0:
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("source1_entity_id\tcandidate_entity_ids\tscores\n")
            for sid in sub_s1["entity_id"]:
                f.write(f"{sid}\t\t\n")
        print(f"  [{country}] no candidates in this country -- wrote all-singleton file")
        return

    t0 = time.time()
    s1_texts = build_composite_text(sub_s1)
    cand_texts = build_composite_text(sub_cand)
    print(f"  [{country}] normalization: {time.time()-t0:.1f}s")

    t_fit = time.time()
    rng = np.random.default_rng(seed)
    if len(cand_texts) > sample_fit_size:
        fit_idx = rng.choice(len(cand_texts), size=sample_fit_size, replace=False)
        fit_sample = [cand_texts[i] for i in fit_idx]
    else:
        fit_sample = cand_texts
    vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 4), max_features=120_000,
        sublinear_tf=True, norm="l2", dtype=np.float32,
    )
    vec.fit(fit_sample)
    print(f"  [{country}] TF-IDF fit: {time.time()-t_fit:.1f}s (vocab={len(vec.vocabulary_):,})")
    del fit_sample

    # --- candidate matrix: chunked ONLY so you get a live bar instead of one 90s+ silent call.
    # Peak memory is unchanged from before: `cand_blocks` is freed before `cand_T` is built,
    # so it's still at most ~2x the candidate matrix size at any instant, same as the original.
    t_tr = time.time()
    cand_blocks = []
    with tqdm(total=n_cands, desc=f"  [{country}] candidate TF-IDF", unit=" cand", leave=False) as pbar:
        for start in range(0, n_cands, cand_transform_chunk):
            end = min(start + cand_transform_chunk, n_cands)
            cand_blocks.append(vec.transform(cand_texts[start:end]))
            pbar.update(end - start)
    mat_cand = sp.vstack(cand_blocks, format="csr")
    del cand_blocks
    cand_T = mat_cand.T.tocsr()
    del cand_texts, mat_cand
    gc.collect()
    print(f"  [{country}] candidate matrix: {time.time()-t_tr:.1f}s | nnz={cand_T.nnz:,}")

    s1_ids = sub_s1["entity_id"].to_numpy()
    cand_ids = sub_cand["entity_id"].to_numpy()

    tmp_path = out_path + ".partial"
    t_srch = time.time()
    with open(tmp_path, "w", encoding="utf-8") as f, \
         tqdm(total=n_queries, desc=f"  [{country}] matching", unit=" query", leave=True) as pbar:
        for start in range(0, n_queries, chunk_size):
            end = min(start + chunk_size, n_queries)
            chunk_t0 = time.time()
            mat_q = vec.transform(s1_texts[start:end])
            # sp_matmul_topn is the current (non-deprecated) API:
            #   ntop -> top_n, lower_bound -> threshold, use_threads/n_jobs -> n_threads
            sim = sp_matmul_topn(mat_q, cand_T, top_n=k, threshold=threshold, n_threads=n_threads, sort=True)
            counts = np.diff(sim.indptr)
            offset = 0
            for i, cnt in enumerate(counts):
                sid = s1_ids[start + i]
                if cnt == 0:
                    f.write(f"{sid}\t\t\n")
                else:
                    row_cands = cand_ids[sim.indices[offset:offset+cnt]]
                    row_scores = sim.data[offset:offset+cnt]
                    f.write(f"{sid}\t{','.join(row_cands.tolist())}\t{','.join(f'{s:.4f}' for s in row_scores)}\n")
                offset += cnt
            del mat_q, sim
            rows_done = end - start
            elapsed = time.time() - chunk_t0
            pbar.set_postfix_str(f"{rows_done/elapsed:,.0f} rows/s")
            pbar.update(rows_done)
    os.rename(tmp_path, out_path)
    print(f"  [{country}] search: {time.time()-t_srch:.1f}s | total: {time.time()-t0:.1f}s")
    del s1_texts, cand_T, vec
    gc.collect()


In [ ]:
print("Loading test_source1...")
test_s1 = pd.read_csv(test_paths["s1"], sep="\t", dtype=str)
test_s1["country"] = test_s1["country"].str.strip()
print(f"  {len(test_s1):,} rows")
print(test_s1["country"].value_counts())

print("\nLoading test candidate pool (S2+S3)...")
t0 = time.time()
test_s2 = pd.read_csv(test_paths["s2"], sep="\t", dtype=str)
test_s3 = pd.read_csv(test_paths["s3"], sep="\t", dtype=str)
test_cand_pool = pd.concat([test_s2, test_s3], ignore_index=True)
test_cand_pool["country"] = test_cand_pool["country"].str.strip()
del test_s2, test_s3
gc.collect()
print(f"  {len(test_cand_pool):,} candidates loaded in {time.time()-t0:.1f}s")

cand_by_country = {c: g.reset_index(drop=True) for c, g in test_cand_pool.groupby("country")}
del test_cand_pool
gc.collect()
s1_by_country = {c: g.reset_index(drop=True) for c, g in test_s1.groupby("country")}

test_country_list = sorted(test_s1["country"].dropna().unique())
overall_t0 = time.time()
for country in tqdm(test_country_list, desc="TEST countries", unit="country"):
    print(f"\n=== TEST -- {country} ===")
    block_one_country(
        country=country,
        sub_s1=s1_by_country[country],
        sub_cand=cand_by_country.get(country, EMPTY_CAND_DF),
        out_path=os.path.join(PARTIAL_DIR, f"test_{country}.tsv"),
    )
print(f"\nAll TEST countries finished in {time.time()-overall_t0:.1f}s total")

del cand_by_country, s1_by_country
gc.collect()
print("\nAll test country partitions blocked -- see partial_test/ for per-country checkpoints.")

Loading test_source1...
  1,732,544 rows
country
India     809986
US        663106
France    259452
Name: count, dtype: int64

Loading test candidate pool (S2+S3)...
  9,969,589 candidates loaded in 34.5s

=== TEST -- France ===
  [France] Queries: 259,452 | Candidate Pool: 1,434,993
  [France] normalization: 29.1s
  [France] TF-IDF fit: 12.7s (vocab=101,432)
  [France] candidate matrix: 93.4s | nnz=118,868,501


In [ ]:
VALIDATION_SAMPLE_PER_COUNTRY = 15_000
VAL_SEED = 7

print("Loading train_source1 + ground truth...")
train_s1_full = pd.read_csv(train_paths["s1"], sep="\t", dtype=str)
train_s1_full["country"] = train_s1_full["country"].str.strip()
gt = pd.read_csv(train_paths["gt"], sep="\t", dtype=str, keep_default_na=False)
gt_map = {
    r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x)
    for r in gt.itertuples(index=False)
}

rng = np.random.default_rng(VAL_SEED)
sampled_ids = []
for country, grp in train_s1_full.groupby("country"):
    n = min(VALIDATION_SAMPLE_PER_COUNTRY, len(grp))
    sampled_ids.extend(rng.choice(grp["entity_id"].to_numpy(), size=n, replace=False))
val_s1 = train_s1_full[train_s1_full["entity_id"].isin(sampled_ids)].reset_index(drop=True)
print(f"  Validation sample: {len(val_s1):,} S1 entities across {val_s1['country'].nunique()} countries")
del train_s1_full
gc.collect()

print("\nLoading + grouping train candidate pool (same size class as test)...")
train_s2 = pd.read_csv(train_paths["s2"], sep="\t", dtype=str)
train_s3 = pd.read_csv(train_paths["s3"], sep="\t", dtype=str)
train_cand_pool = pd.concat([train_s2, train_s3], ignore_index=True)
train_cand_pool["country"] = train_cand_pool["country"].str.strip()
del train_s2, train_s3
gc.collect()
train_cand_by_country = {c: g.reset_index(drop=True) for c, g in train_cand_pool.groupby("country")}
del train_cand_pool
gc.collect()

val_country_list = sorted(val_s1["country"].dropna().unique())
val_overall_t0 = time.time()
for country in tqdm(val_country_list, desc="VALIDATION countries", unit="country"):
    print(f"\n=== VALIDATION -- {country} ===")
    block_one_country(
        country=country,
        sub_s1=val_s1[val_s1["country"] == country].reset_index(drop=True),
        sub_cand=train_cand_by_country.get(country, EMPTY_CAND_DF),
        out_path=os.path.join(VAL_PARTIAL_DIR, f"val_{country}.tsv"),
    )
print(f"\nAll VALIDATION countries finished in {time.time()-val_overall_t0:.1f}s total")
del train_cand_by_country
gc.collect()

In [ ]:
def load_all_partials(directory, prefix):
    rows = {}
    for fn in sorted(os.listdir(directory)):
        if not (fn.startswith(prefix) and fn.endswith(".tsv")):
            continue
        df = pd.read_csv(os.path.join(directory, fn), sep="\t", dtype=str, keep_default_na=False)
        for r in df.itertuples(index=False):
            cids = [x for x in r.candidate_entity_ids.split(",") if x]
            scores = [float(x) for x in r.scores.split(",") if x]
            rows[r.source1_entity_id] = list(zip(cids, scores))
    return rows

def f_beta(true_ids, pred_ids, beta=0.5):
    true_ids, pred_ids = set(true_ids), set(pred_ids)
    if not true_ids:
        return 1.0 if not pred_ids else 0.0
    if not pred_ids:
        return 0.0
    tp = len(true_ids & pred_ids)
    p = tp / len(pred_ids)
    r = tp / len(true_ids)
    if p == 0 and r == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * p * r / (b2 * p + r)

val_results = load_all_partials(VAL_PARTIAL_DIR, "val_")
val_ids = list(val_results.keys())

recall_hits, recall_total = 0, 0
for sid in val_ids:
    true = gt_map.get(sid, set())
    if not true:
        continue
    all_cands = {cid for cid, _ in val_results[sid]}
    recall_hits += len(true & all_cands)
    recall_total += len(true)
print(f"Blocking recall ceiling (k=25, before any threshold): "
      f"{recall_hits}/{recall_total} = {recall_hits/max(recall_total,1):.3f}")
print("(a match the blocker never retrieved can't be recovered by any threshold")
print(" or later matching model -- this is your real ceiling)")

print(f"\n{'threshold':>10} | {'macro F0.5':>10}")
best_tau, best_f = None, -1.0
for tau in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    scores = [f_beta(gt_map.get(sid, set()), [cid for cid, sc in val_results[sid] if sc >= tau])
              for sid in val_ids]
    macro_f = float(np.mean(scores))
    print(f"{tau:>10.2f} | {macro_f:>10.4f}")
    if macro_f > best_f:
        best_f, best_tau = macro_f, tau

print(f"\nBest threshold on validation sample: tau={best_tau} (macro F0.5={best_f:.4f})")

In [ ]:
FINAL_THRESHOLD = best_tau if "best_tau" in dir() else 0.45
print(f"Using precision threshold tau = {FINAL_THRESHOLD}")

test_results = load_all_partials(PARTIAL_DIR, "test_")
all_test_ids = test_s1["entity_id"].tolist()

cand_file = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
match_file = os.path.join(OUTPUT_DIR, "matching_results.tsv")

with open(cand_file, "w", encoding="utf-8") as fc, open(match_file, "w", encoding="utf-8") as fm:
    fc.write("source1_entity_id\tcandidate_entity_ids\n")
    fm.write("source1_entity_id\tmatched_entity_ids\n")
    n_singleton = 0
    for sid in all_test_ids:
        pairs = test_results.get(sid, [])
        fc.write(f"{sid}\t{','.join(cid for cid, _ in pairs)}\n")
        matched = [cid for cid, sc in pairs if sc >= FINAL_THRESHOLD]
        if not matched:
            n_singleton += 1
        fm.write(f"{sid}\t{','.join(matched)}\n")

print(f"Wrote {cand_file} and {match_file}")
print(f"  {n_singleton:,}/{len(all_test_ids):,} predicted as singletons "
      f"({n_singleton/len(all_test_ids):.1%})")

In [ ]:
def run_local_validation(matching_path, candidate_path, test_s1_path, test_cand_ids):
    errors = []
    print("\nRunning comprehensive validation checks...")

    s1_ids = pd.read_csv(test_s1_path, sep="\t", dtype=str)["entity_id"].tolist()
    expected_rows = len(s1_ids)

    df_m = pd.read_csv(matching_path, sep="\t", dtype=str).fillna("")
    df_c = pd.read_csv(candidate_path, sep="\t", dtype=str).fillna("")

    if len(df_m) != expected_rows:
        errors.append(f"matching_results.tsv row count mismatch: {len(df_m)} vs {expected_rows}")
    if len(df_c) != expected_rows:
        errors.append(f"candidate_pairs.tsv row count mismatch: {len(df_c)} vs {expected_rows}")

    if list(df_m.columns) != ["source1_entity_id", "matched_entity_ids"]:
        errors.append(f"Invalid columns in matching_results: {list(df_m.columns)}")
    if list(df_c.columns) != ["source1_entity_id", "candidate_entity_ids"]:
        errors.append(f"Invalid columns in candidate_pairs: {list(df_c.columns)}")

    m_dict = dict(zip(df_m["source1_entity_id"], df_m["matched_entity_ids"]))
    c_dict = dict(zip(df_c["source1_entity_id"], df_c["candidate_entity_ids"]))

    for s1 in s1_ids:
        m_list = [x.strip() for x in m_dict.get(s1, "").split(",") if x.strip()]
        c_list = [x.strip() for x in c_dict.get(s1, "").split(",") if x.strip()]

        if len(m_list) != len(set(m_list)):
            errors.append(f"Duplicate IDs in match list for {s1}"); break
        if len(c_list) != len(set(c_list)):
            errors.append(f"Duplicate IDs in candidate list for {s1}"); break
        if not set(m_list).issubset(set(c_list)):
            errors.append(f"Match ID for {s1} is not present in its candidate list!"); break
        for cid in c_list:
            if not (cid.startswith("S2-") or cid.startswith("S3-")):
                errors.append(f"Candidate {cid} does not have S2- or S3- prefix"); break
            if cid not in test_cand_ids:
                errors.append(f"Entity {cid} does not exist in candidate pool!"); break
        if errors:
            break

    if not errors:
        print("=" * 60)
        print("STATUS: PASS (EXIT 0) -- READY FOR PORTAL SUBMISSION!")
        print("=" * 60)
    else:
        print("STATUS: FAILED")
        for err in errors:
            print(f"  [ERROR] {err}")

_test_cand_ids = set()
for p in (test_paths["s2"], test_paths["s3"]):
    _test_cand_ids.update(pd.read_csv(p, sep="\t", dtype=str, usecols=["entity_id"])["entity_id"])

run_local_validation(
    matching_path=match_file,
    candidate_path=cand_file,
    test_s1_path=test_paths["s1"],
    test_cand_ids=_test_cand_ids,
)

official_script = "student_resource/utils/validate_submission.py"
if os.path.exists(official_script):
import sys
    get_ipython().system('{sys.executable} {official_script} --matching {match_file} --candidate {cand_file} --test-dir {TEST_DIR}')